# 🧠 Deep Learning EEG Seizure Detection v3 (Anti-Overfitting)
**Dataset:** CHB-MIT Scalp EEG Database (Kaggle)
**Architecture:** 1D-CNN + BiGRU + Temporal Attention
**Key Features:**
- Bandpass Filter (0.5-40Hz) → loại bỏ nhiễu điện lưới & cơ
- Asymmetric Overlapping → nhân bản Seizure x4 tự nhiên
- Patient-Aware Split → chống Data Leakage
- Focal Loss + Online Augmentation → xử lý mất cân bằng cực đại
- Mixed Precision (AMP) + DataParallel → tối ưu T4x2 GPU

**v3 Anti-Overfitting Upgrades (so với v2):**
- ⬇️ LR 1e-3 → 3e-4 (giảm LR tối đa của OneCycleLR)
- ⬆️ Weight Decay 1e-4 → 1e-2 (L2 penalty mạnh hơn)
- ⬆️ Epochs 30 → 50 (LR thấp hơn cần nhiều epoch hơn để hội tụ)
- ➕ Spatial Dropout sau CNN blocks (giảm co-adaptation)
- ➕ 2-layer BiGRU với dropout 0.3 (tăng capacity + regularization)
- ➕ Label Smoothing trong Focal Loss
- ➕ Early Stopping (patience=10) chống train quá lâu
- ➕ CosineAnnealingWarmRestarts thay OneCycleLR (linh hoạt hơn)


In [ ]:
# CELL 1: Cài thư viện
!pip install mne pyedflib scikit-learn h5py matplotlib seaborn tqdm -q
print('✅ Cài thư viện thành công!')

In [ ]:
# CELL 2: Import thư viện
import inspect
import os
import re
import numpy as np
import pandas as pd
import mne
import warnings
import h5py
import random
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal as sp_signal

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
import pyedflib
from tqdm.auto import tqdm
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)
import gc
import time

warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

print('✅ Import thành công!')
print(f'🔥 PyTorch: {torch.__version__}')
print(f'🔥 CUDA: {torch.cuda.is_available()} | GPU count: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'   GPU {i}: {torch.cuda.get_device_name(i)}')
print(f'🔥 CPU cores: {os.cpu_count()}')

In [ ]:
# CELL 3: Cấu hình Pipeline (v3 - Anti-Overfitting)
DATASET_PATH = '/kaggle/input/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0'
OUTPUT_PATH = '/kaggle/working'
H5_FILE = os.path.join(OUTPUT_PATH, 'eeg_data.h5')

# ============ EEG Config ============
SAMPLING_RATE = 256
WINDOW_SIZE = 4                          # giây
WINDOW_SAMPLES = SAMPLING_RATE * WINDOW_SIZE  # 1024

MAX_CHANNELS = 23
MIN_EEG_CHANNELS = 5
DTYPE = np.float32

# ============ Bandpass Filter ============
BANDPASS_LOW = 0.5    # Hz
BANDPASS_HIGH = 40.0  # Hz
FILTER_ORDER = 4      # Butterworth order

# ============ Asymmetric Overlap ============
NORMAL_OVERLAP = 0.0     # 0% overlap cho Normal
SEIZURE_OVERLAP = 0.75   # 75% overlap cho Seizure (x4 augmentation tự nhiên)
NORMAL_KEEP_PROB = 0.15  # Giữ 15% Normal windows

# ============ Training (v3 ANTI-OVERFITTING) ============
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 256
EPOCHS = 50              # ⬆️ 30→50: LR thấp cần nhiều epoch hơn để hội tụ
LR = 3e-4                # ⬇️ 1e-3→3e-4: giảm LR max, chống dao động Val Loss
WEIGHT_DECAY = 1e-2      # ⬆️ 1e-4→1e-2: L2 penalty mạnh hơn, chống học vẹt
LABEL_SMOOTHING = 0.05   # ➕ Mới: làm mượt label để model không overconfident
EARLY_STOPPING_PATIENCE = 10  # ➕ Mới: dừng sớm nếu Val F1 không cải thiện

# ============ Kaggle I/O + GPU tuning ============
H5_COMPRESSION = None    # ⚡ BỎ NÉN → ghi/đọc nhanh hơn rất nhiều
H5_CHUNK_SIZE = 256      # ⚡ Chunk lớn hơn → sequential read nhanh hơn
NUM_WORKERS = 2          # ⚡ 2 là sweet spot trên Kaggle (tránh memory overhead)
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 4 if NUM_WORKERS > 0 else None
PIN_MEMORY = DEVICE.type == 'cuda'
USE_WEIGHTED_SAMPLER = False
USE_DATA_PARALLEL = True  # ⚡ BẬT: dùng cả 2 GPU T4
ENABLE_BATCH_AUGMENTATION = True

# ============ Multiprocessing cho Cell 6 ============
MAX_EXTRACT_WORKERS = min(4, os.cpu_count() or 2)  # ⚡ Song song hóa đọc EDF

# ============ Focal Loss ============
FOCAL_ALPHA = 0.75  # Trọng số cho lớp Seizure (thiểu số)
FOCAL_GAMMA = 2.0   # Mức độ phạt các mẫu dễ đoán

MAX_PATIENTS = None  # None = dùng toàn bộ

print('📋 Cấu hình Pipeline v3 (Anti-Overfitting):')
print(f'   Device          : {DEVICE}')
print(f'   Window          : {WINDOW_SIZE}s ({WINDOW_SAMPLES} samples)')
print(f'   Bandpass        : {BANDPASS_LOW}-{BANDPASS_HIGH} Hz')
print(f'   Seizure Overlap : {SEIZURE_OVERLAP*100:.0f}%')
print(f'   Normal Keep     : {NORMAL_KEEP_PROB*100:.0f}%')
print(f'   Batch Size      : {BATCH_SIZE}')
print(f'   Epochs          : {EPOCHS}')
print(f'   LR (max)        : {LR}')
print(f'   Weight Decay    : {WEIGHT_DECAY}')
print(f'   Label Smoothing : {LABEL_SMOOTHING}')
print(f'   Early Stopping  : patience={EARLY_STOPPING_PATIENCE}')
print(f'   H5 Compression  : {H5_COMPRESSION} (disabled for speed)')
print(f'   Workers         : {NUM_WORKERS}')
print(f'   Pin Memory      : {PIN_MEMORY}')
print(f'   Data Parallel   : {USE_DATA_PARALLEL}')
print(f'   Focal Loss      : alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA}')


In [ ]:
# CELL 4: Tìm dataset + Đọc annotations
import os, re

# === Danh sách path có thể chứa dataset ===
CANDIDATE_PATHS = [
    # Path từ Cell 3 (nếu đã chạy)
    globals().get('DATASET_PATH', ''),
    # Các slug phổ biến trên Kaggle
    '/kaggle/input/seizure-epilepcy-chb-mit-eeg-dataset-pediatric/chb-mit-scalp-eeg-database-1.0.0',
    '/kaggle/input/chb-mit-scalp-eeg-database',
    '/kaggle/input/chbmit-scalp-eeg-database',
    '/kaggle/input/chb-mit-eeg',
]

# Bước 1: Thử từng path ứng viên
DATASET_PATH = None
for p in CANDIDATE_PATHS:
    if p and isinstance(p, str) and os.path.isdir(p):
        # Kiểm tra có chứa thư mục chbXX không
        if any(re.match(r'^chb\d+$', d) for d in os.listdir(p) if os.path.isdir(os.path.join(p, d))):
            DATASET_PATH = p
            break

# Bước 2: Fallback - quét toàn bộ /kaggle/input
if DATASET_PATH is None:
    print('🔍 Không tìm thấy ở path mặc định, quét /kaggle/input...')
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(re.match(r'^chb\d+$', d) for d in dirs):
            DATASET_PATH = root
            break

# Bước 3: Fallback - tìm file .edf
if DATASET_PATH is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.edf') for f in files):
            DATASET_PATH = os.path.dirname(root)
            break

# Bước 4: Thất bại - in cấu trúc thư mục để debug
if DATASET_PATH is None:
    print('\n' + '='*60)
    print('❌ KHÔNG TÌM THẤY DATASET CHB-MIT!')
    print('='*60)
    print('\n📂 Cấu trúc /kaggle/input:')
    for root, dirs, files in os.walk('/kaggle/input'):
        depth = root.replace('/kaggle/input', '').count(os.sep)
        if depth <= 3:
            indent = '  ' * depth
            print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files)')
            if depth <= 2:
                for d in sorted(dirs)[:5]:
                    print(f'{indent}  📁 {d}/')
                if len(dirs) > 5:
                    print(f'{indent}  ... và {len(dirs)-5} thư mục khác')
    print('\n💡 Hướng dẫn: Vào Kaggle notebook → Add Data → Search "CHB-MIT"')
    print('   Dataset: https://www.kaggle.com/datasets/shoeb28/seizure-epilepcy-chb-mit-eeg-dataset-pediatric')
    raise FileNotFoundError('Dataset CHB-MIT chưa được add vào notebook!')

print(f'✅ Dataset: {DATASET_PATH}')

def parse_summary_file(summary_path):
    seizure_info = {}
    if not os.path.exists(summary_path):
        return seizure_info
    with open(summary_path, 'r') as f:
        content = f.read()
    blocks = re.split(r'File Name:\s*', content)
    for block in blocks[1:]:
        lines = block.strip().split('\n')
        filename = lines[0].strip()
        num_match = re.search(r'Number of Seizures in File:\s*(\d+)', block)
        if not num_match:
            continue
        num_seizures = int(num_match.group(1))
        if num_seizures == 0:
            seizure_info[filename] = []
            continue
        seizures = []
        starts = re.findall(r'Seizure\s*\d*\s*Start Time:\s*(\d+)\s*seconds', block)
        ends = re.findall(r'Seizure\s*\d*\s*End Time:\s*(\d+)\s*seconds', block)
        for s, e in zip(starts, ends):
            seizures.append((int(s), int(e)))
        seizure_info[filename] = seizures
    return seizure_info

patient_dirs = sorted([
    d for d in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, d)) and re.match(r'^chb\d+$', d)
])
print(f'📁 Số bệnh nhân: {len(patient_dirs)}')
print(f'   Danh sách: {patient_dirs}')

In [ ]:
# CELL 5: Bandpass Filter (VECTORIZED) + Khởi tạo HDF5

def create_bandpass_filter(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    sos = sp_signal.butter(order, [low, high], btype='band', output='sos')
    return sos

def apply_bandpass(data, sos):
    """
    Bandpass filter VECTORIZED - xu ly tat ca channels cung luc.
    data shape: (n_channels, n_samples)
    Dung axis=1 de scipy xu ly noi bo - nhanh gap 5-10x.
    """
    return sp_signal.sosfiltfilt(sos, data, axis=1).astype(np.float32)

BANDPASS_SOS = create_bandpass_filter(BANDPASS_LOW, BANDPASS_HIGH, SAMPLING_RATE, FILTER_ORDER)
print(f'✅ Butterworth Bandpass: {BANDPASS_LOW}-{BANDPASS_HIGH} Hz (VECTORIZED)')

if os.path.exists(H5_FILE):
    os.remove(H5_FILE)
    print(f'🗑️ Đã xóa: {H5_FILE}')

with h5py.File(H5_FILE, 'w') as f:
    f.create_dataset('X', shape=(0, MAX_CHANNELS, WINDOW_SAMPLES),
                     maxshape=(None, MAX_CHANNELS, WINDOW_SAMPLES),
                     dtype=DTYPE,
                     chunks=(min(256, BATCH_SIZE), MAX_CHANNELS, WINDOW_SAMPLES))
    f.create_dataset('y', shape=(0,), maxshape=(None,),
                     dtype=np.uint8, chunks=True)
    f.create_dataset('patient_ids', shape=(0,), maxshape=(None,),
                     dtype=h5py.special_dtype(vlen=str), chunks=True)
print(f'📁 HDF5 sẵn sàng: {H5_FILE}')

In [ ]:
# CELL 6: Trích xuất dữ liệu - VECTORIZED (không multiprocessing)
# Nhanh hon multiprocessing tren Kaggle NFS vi tranh overhead serialize

import time

NORMAL_STEP = int(WINDOW_SAMPLES * (1 - NORMAL_OVERLAP))
SEIZURE_STEP = int(WINDOW_SAMPLES * (1 - SEIZURE_OVERLAP))  # 256 = 1s

total_windows = 0
total_seizure = 0
total_normal = 0

patients_to_process = patient_dirs[:MAX_PATIENTS] if MAX_PATIENTS else patient_dirs

t_start = time.time()

# Count EDF files
all_edf_count = 0
for p in patients_to_process:
    pp = os.path.join(DATASET_PATH, p)
    all_edf_count += len([f for f in os.listdir(pp) if f.endswith('.edf')])
print(f'📋 Tổng: {all_edf_count} file EDF từ {len(patients_to_process)} bệnh nhân')

pbar = tqdm(total=all_edf_count, desc='⚡ Trích xuất')
write_buf_X, write_buf_y, write_buf_pid = [], [], []
buf_count = 0

def flush_h5():
    global write_buf_X, write_buf_y, write_buf_pid, buf_count
    global total_windows, total_seizure, total_normal
    if buf_count == 0:
        return
    X_all = np.concatenate(write_buf_X, axis=0)
    y_all = np.concatenate(write_buf_y, axis=0)
    with h5py.File(H5_FILE, 'a') as f:
        curr = f['X'].shape[0]
        n = len(X_all)
        f['X'].resize((curr + n), axis=0)
        f['y'].resize((curr + n), axis=0)
        f['patient_ids'].resize((curr + n), axis=0)
        f['X'][curr:] = X_all
        f['y'][curr:] = y_all
        f['patient_ids'][curr:] = write_buf_pid
    total_seizure += int(y_all.sum())
    total_normal += int((y_all == 0).sum())
    total_windows += n
    write_buf_X.clear()
    write_buf_y.clear()
    write_buf_pid.clear()
    buf_count = 0

for patient in patients_to_process:
    patient_path = os.path.join(DATASET_PATH, patient)
    summary_path = os.path.join(patient_path, f'{patient}-summary.txt')
    seizure_info = parse_summary_file(summary_path)
    edf_files = sorted([f for f in os.listdir(patient_path) if f.endswith('.edf')])

    for edf_file in edf_files:
        edf_path = os.path.join(patient_path, edf_file)
        reader = None
        data = None

        try:
            reader = pyedflib.EdfReader(edf_path)
            labels_edf = reader.getSignalLabels()
            sf = int(reader.getSampleFrequency(0))
            n_samples = int(reader.getNSamples()[0])

            eeg_idx = [i for i, lb in enumerate(labels_edf)
                       if lb.strip() not in ('-', '')
                       and 'ECG' not in lb.upper()
                       and 'VNS' not in lb.upper()]
            if len(eeg_idx) < MIN_EEG_CHANNELS:
                pbar.update(1)
                continue
            eeg_idx = eeg_idx[:MAX_CHANNELS]

            data = np.stack([reader.readSignal(i).astype(DTYPE) for i in eeg_idx], axis=0)

            if data.shape[0] < MAX_CHANNELS:
                pad = np.zeros((MAX_CHANNELS - data.shape[0], data.shape[1]), dtype=DTYPE)
                data = np.vstack([data, pad])

            # VECTORIZED bandpass (tat ca channels cung luc)
            data = apply_bandpass(data, BANDPASS_SOS)

            file_seizures = seizure_info.get(edf_file, [])
            file_X, file_y = [], []

            # Seizure mask
            seizure_mask = np.zeros(n_samples, dtype=bool)
            for sz_s, sz_e in file_seizures:
                seizure_mask[int(sz_s * sf):min(int(sz_e * sf), n_samples)] = True

            # ===== SEIZURE WINDOWS (overlap 75%) =====
            for sz_s, sz_e in file_seizures:
                sz_start = max(0, int(sz_s * sf) - WINDOW_SAMPLES)
                sz_end = min(n_samples, int(sz_e * sf) + WINDOW_SAMPLES)
                starts = np.arange(sz_start, sz_end - WINDOW_SAMPLES, SEIZURE_STEP, dtype=np.int64)
                if len(starts) == 0:
                    continue
                # Vectorized overlap check
                start_secs = starts / sf
                end_secs = (starts + WINDOW_SAMPLES) / sf
                overlap_sec = np.minimum(end_secs, sz_e) - np.maximum(start_secs, sz_s)
                valid = overlap_sec / WINDOW_SIZE > 0.5
                for s in starts[valid]:
                    w = data[:, s:s+WINDOW_SAMPLES]
                    if w.shape[1] == WINDOW_SAMPLES:
                        file_X.append(w)
                        file_y.append(1)

            # ===== NORMAL WINDOWS (overlap 0%, giu 15%) =====
            starts_normal = np.arange(0, n_samples - WINDOW_SAMPLES, NORMAL_STEP, dtype=np.int64)
            if len(starts_normal) > 0:
                # Vectorized seizure ratio via cumsum
                cs = np.cumsum(np.concatenate(([0], seizure_mask.astype(np.float32))))
                sz_sums = cs[starts_normal + WINDOW_SAMPLES] - cs[starts_normal]
                sz_ratios = sz_sums / WINDOW_SAMPLES
                ok_mask = sz_ratios <= 0.2
                rnd_keep = np.random.random(len(starts_normal)) <= NORMAL_KEEP_PROB
                keep = ok_mask & rnd_keep
                for s in starts_normal[keep]:
                    w = data[:, s:s+WINDOW_SAMPLES]
                    if w.shape[1] == WINDOW_SAMPLES:
                        file_X.append(w)
                        file_y.append(0)

            if len(file_X) > 0:
                write_buf_X.append(np.array(file_X, dtype=DTYPE))
                write_buf_y.append(np.array(file_y, dtype=np.uint8))
                write_buf_pid.extend([patient] * len(file_X))
                buf_count += len(file_X)

            if buf_count >= 5000:
                flush_h5()

        except Exception as e:
            if 'cannot' not in str(e).lower():
                print(f'⚠️ {patient}/{edf_file}: {e}')
        finally:
            if reader is not None:
                try: reader.close()
                except: pass
            del data
            gc.collect()

        pbar.update(1)
        pbar.set_postfix_str(f'W:{total_windows+buf_count}|Sz:{total_seizure}')

flush_h5()
pbar.close()

elapsed = time.time() - t_start
print(f'\n🎉 XONG! Tổng: {total_windows} windows')
print(f'   Seizure : {total_seizure}')
print(f'   Normal  : {total_normal}')
print(f'   Tỉ lệ   : 1:{total_normal/max(total_seizure,1):.0f}')
print(f'   ⏱️ Thời gian: {elapsed:.0f}s ({elapsed/60:.1f} phút)')

gc.collect()

In [ ]:
# CELL 7: PyTorch Dataset với Online Augmentation + Patient-Aware Split

class EEGDatasetH5(Dataset):
    """
    Lazy-load raw EEG windows từ HDF5.
    Z-score và augmentation sẽ xử lý theo batch bằng torch để giảm nghẽn CPU.
    """
    def __init__(self, h5_path, indices):
        self.h5_path = h5_path
        self.indices = np.asarray(indices, dtype=np.int64)
        self._h5 = None
        self._x = None
        self._y = None

    def _open(self):
        if self._h5 is None:
            self._h5 = h5py.File(
                self.h5_path,
                'r',
                rdcc_nbytes=512 * 1024 * 1024,   # ⚡ 512MB cache (tăng từ 256MB)
                rdcc_nslots=1_000_003
            )
            self._x = self._h5['X']
            self._y = self._h5['y']

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        self._open()
        real_idx = int(self.indices[idx])
        x = np.asarray(self._x[real_idx], dtype=np.float32)
        y = np.float32(self._y[real_idx])
        return x, y

    def __del__(self):
        try:
            if self._h5 is not None:
                self._h5.close()
        except Exception:
            pass

# ========== Patient-Aware Split ==========
print('🔬 Patient-Aware Split...')

with h5py.File(H5_FILE, 'r') as f:
    total_samples = f['X'].shape[0]
    all_y = f['y'][:]
    all_pids = f['patient_ids'][:]
    if isinstance(all_pids[0], bytes):
        all_pids = np.array([p.decode() for p in all_pids])

# Chia theo bệnh nhân: 80% train, 20% test
unique_patients = sorted(set(all_pids))
np.random.seed(SEED)
np.random.shuffle(unique_patients)

n_train_patients = int(0.8 * len(unique_patients))
train_patients = set(unique_patients[:n_train_patients])
test_patients = set(unique_patients[n_train_patients:])

train_idx = np.array([i for i in range(total_samples) if all_pids[i] in train_patients])
test_idx = np.array([i for i in range(total_samples) if all_pids[i] in test_patients])

y_train = all_y[train_idx]
y_test = all_y[test_idx]

print(f'\n✅ Patient-Aware Split thành công!')
print(f'   Train patients: {sorted(train_patients)}')
print(f'   Test patients : {sorted(test_patients)}')
print(f'   Train size    : {len(train_idx)} (Seizure: {(y_train==1).sum()}, Normal: {(y_train==0).sum()})')
print(f'   Test size     : {len(test_idx)} (Seizure: {(y_test==1).sum()}, Normal: {(y_test==0).sum()})')

# DataLoader
train_dataset = EEGDatasetH5(H5_FILE, train_idx)
test_dataset = EEGDatasetH5(H5_FILE, test_idx)

class_counts = np.array([int((y_train == 0).sum()), int((y_train == 1).sum())])
weight_per_class = 1.0 / np.maximum(class_counts, 1)
sample_weights = np.array([weight_per_class[int(y)] for y in y_train], dtype=np.float64)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=True,
    persistent_workers=PERSISTENT_WORKERS
)
test_loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS
)
if NUM_WORKERS > 0:
    train_loader_kwargs['prefetch_factor'] = PREFETCH_FACTOR
    test_loader_kwargs['prefetch_factor'] = PREFETCH_FACTOR

if USE_WEIGHTED_SAMPLER:
    sampler = WeightedRandomSampler(
        torch.from_numpy(sample_weights).double(),
        num_samples=len(sample_weights),
        replacement=True
    )
    train_loader = DataLoader(
        train_dataset,
        sampler=sampler,
        shuffle=False,
        **train_loader_kwargs
    )
else:
    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        generator=loader_generator,
        **train_loader_kwargs
    )

test_loader = DataLoader(
    test_dataset,
    shuffle=False,
    generator=loader_generator,
    **test_loader_kwargs
)

print(f'   Batches/epoch : {len(train_loader)}')
print(f'   DataLoader    : workers={NUM_WORKERS}, pin_memory={PIN_MEMORY}, prefetch={PREFETCH_FACTOR}')
print(f'   Sampler       : {"WeightedRandomSampler" if USE_WEIGHTED_SAMPLER else "Shuffle"}')

In [ ]:
# CELL 8: Kiến trúc 1D-CNN + BiGRU + Temporal Attention + Focal Loss (v3)

# ==================== FOCAL LOSS + LABEL SMOOTHING ====================
class FocalLoss(nn.Module):
    """
    Focal Loss v3: thêm Label Smoothing giảm overconfidence.
    - alpha: trọng số cho lớp thiểu số (Seizure)
    - gamma: mức độ giảm weight cho mẫu dễ đoán
    - label_smoothing: làm mượt nhãn (0.05 = target 0→0.025, 1→0.975)
    """
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        # Label smoothing: 0→ε/2, 1→1-ε/2
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)  # probability of correct class
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - pt) ** self.gamma
        return (focal_weight * bce).mean()

# ==================== TEMPORAL ATTENTION ====================
class TemporalAttention(nn.Module):
    """
    Soft attention trên chiều thời gian.
    Học cách tập trung vào vùng sóng bất thường.
    """
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, gru_output):
        # gru_output: (batch, seq_len, hidden*2)
        scores = self.attention(gru_output).squeeze(-1)  # (batch, seq_len)
        weights = F.softmax(scores, dim=1).unsqueeze(-1)  # (batch, seq_len, 1)
        context = (gru_output * weights).sum(dim=1)        # (batch, hidden*2)
        return context, weights.squeeze(-1)

# ==================== MAIN MODEL (v3 - ANTI-OVERFITTING) ====================
class EEG_CNN_BiGRU_Attention(nn.Module):
    """
    Hybrid Architecture v3 (Anti-Overfitting):
    1. CNN blocks + Spatial Dropout: trích xuất mẫu sóng cục bộ, giảm co-adaptation
    2. 2-layer BiGRU + dropout: nắm bắt ngữ cảnh thời gian sâu hơn
    3. Temporal Attention: tập trung vào vùng bất thường
    4. Classifier + Dropout: phân loại với regularization mạnh
    """
    def __init__(self, in_channels=23, gru_hidden=64, num_classes=1):
        super().__init__()

        # === CNN Feature Extractor + Spatial Dropout ===
        # Block 1: (23, 1024) -> (32, 256)
        self.conv_block1 = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2)
        )
        self.drop1 = nn.Dropout(0.1)  # ➕ Spatial dropout sau block 1

        # Block 2: (32, 256) -> (64, 64)
        self.conv_block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2)
        )
        self.drop2 = nn.Dropout(0.2)  # ➕ Spatial dropout sau block 2

        # Block 3: (64, 64) -> (128, 64)
        self.conv_block3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True)
        )
        self.drop3 = nn.Dropout(0.2)  # ➕ Spatial dropout sau block 3

        # === BiGRU Temporal Encoder (v3: 2 layers + dropout) ===
        self.gru = nn.GRU(
            input_size=128,
            hidden_size=gru_hidden,
            num_layers=2,        # ⬆️ 1→2 layers: tăng capacity
            batch_first=True,
            bidirectional=True,
            dropout=0.3          # ⬆️ 0.0→0.3: dropout giữa 2 GRU layers
        )

        # === Temporal Attention ===
        self.attention = TemporalAttention(gru_hidden * 2)

        # === Classifier (giữ dropout 0.5 + 0.3 như v2) ===
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(gru_hidden * 2, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # x: (Batch, 23, 1024)

        # CNN + Spatial Dropout
        x = self.drop1(self.conv_block1(x))   # (B, 32, 256)
        x = self.drop2(self.conv_block2(x))   # (B, 64, 64)
        x = self.drop3(self.conv_block3(x))   # (B, 128, 64)

        # Reshape: (B, 128, 64) -> (B, 64, 128) cho GRU
        x = x.permute(0, 2, 1)    # (B, seq_len=64, features=128)

        # 2-layer BiGRU
        gru_out, _ = self.gru(x)   # (B, 64, hidden*2=128)

        # Attention
        context, attn_weights = self.attention(gru_out)  # (B, hidden*2)

        # Classify
        out = self.classifier(context)  # (B, 1)
        return out

# ==================== BUILD MODEL ====================
model = EEG_CNN_BiGRU_Attention(in_channels=MAX_CHANNELS).to(DEVICE)

# ⚡ torch.compile nếu PyTorch >= 2.0 (tăng tốc 10-30%)
USE_COMPILE = False
if hasattr(torch, 'compile') and DEVICE.type == 'cuda':
    try:
        model = torch.compile(model, mode='reduce-overhead')
        USE_COMPILE = True
        print('⚡ torch.compile enabled (reduce-overhead mode)')
    except Exception as e:
        print(f'⚠️ torch.compile failed: {e}, using eager mode')

use_data_parallel = (
    USE_DATA_PARALLEL and
    DEVICE.type == 'cuda' and
    torch.cuda.device_count() > 1
)
if use_data_parallel:
    print(f'🚀 DataParallel: {torch.cuda.device_count()} GPUs!')
    model = nn.DataParallel(model)
elif DEVICE.type == 'cuda':
    print(f'⚡ Single-GPU mode on: {torch.cuda.get_device_name(0)}')
else:
    print('⚡ CPU mode')

def get_model_state_dict(model):
    return model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()

def load_model_checkpoint(model, checkpoint_path, device):
    state_dict = torch.load(checkpoint_path, map_location=device)
    first_key = next(iter(state_dict))
    has_module_prefix = first_key.startswith('module.')

    if isinstance(model, nn.DataParallel) and not has_module_prefix:
        state_dict = {f'module.{k}': v for k, v in state_dict.items()}
    elif not isinstance(model, nn.DataParallel) and has_module_prefix:
        state_dict = {k.replace('module.', '', 1): v for k, v in state_dict.items()}

    model.load_state_dict(state_dict)
    return model

# ==================== LOSS + OPTIMIZER + SCHEDULER (v3) ====================
criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)

# ⬇️ Dùng AdamW thay Adam: tách biệt weight decay khỏi gradient → hiệu quả hơn
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# ⬇️ CosineAnnealingWarmRestarts: cho phép LR giảm rồi tăng lại theo chu kỳ
# T_0=10 epoch mỗi chu kỳ, T_mult=2 (chu kỳ sau dài gấp đôi)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

# Mixed Precision Scaler
scaler = GradScaler(enabled=DEVICE.type == 'cuda')

# In thông tin model
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n✅ Model v3: EEG_CNN_BiGRU_Attention (Anti-Overfitting)')
print(f'   Total params    : {total_params:,}')
print(f'   Trainable params: {trainable_params:,}')
print(f'   Loss            : Focal Loss (alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA}, smoothing={LABEL_SMOOTHING})')
print(f'   Optimizer       : AdamW (lr={LR}, wd={WEIGHT_DECAY})')
print(f'   Scheduler       : CosineAnnealingWarmRestarts (T0=10, Tmult=2)')
print(f'   Mixed Precision : {DEVICE.type == "cuda"}')
print(f'   Data Parallel   : {use_data_parallel}')
print(f'   torch.compile   : {USE_COMPILE}')


In [ ]:
# CELL 9: Hàm Train và Eval với Mixed Precision (v3)

def augment_batch(X_batch):
    batch_size, n_channels, _ = X_batch.shape

    noise_mask = torch.rand(batch_size, device=X_batch.device) < 0.5
    if noise_mask.any():
        X_batch[noise_mask] += 0.01 * torch.randn_like(X_batch[noise_mask])

    scale_mask = torch.rand(batch_size, device=X_batch.device) < 0.5
    if scale_mask.any():
        n_scale = int(scale_mask.sum().item())
        scales = torch.empty(n_scale, 1, 1, device=X_batch.device, dtype=X_batch.dtype).uniform_(0.8, 1.2)
        X_batch[scale_mask] *= scales

    channel_mask_samples = torch.rand(batch_size, device=X_batch.device) < 0.3
    if channel_mask_samples.any():
        n_mask = torch.randint(1, 3, (batch_size,), device=X_batch.device)
        channel_scores = torch.rand(batch_size, n_channels, device=X_batch.device)
        channel_order = channel_scores.argsort(dim=1)
        channel_mask = torch.zeros(batch_size, n_channels, dtype=torch.bool, device=X_batch.device)
        batch_indices = torch.arange(batch_size, device=X_batch.device)
        channel_mask[batch_indices, channel_order[:, 0]] = channel_mask_samples
        channel_mask[batch_indices, channel_order[:, 1]] = channel_mask_samples & (n_mask == 2)
        X_batch = X_batch.masked_fill(channel_mask.unsqueeze(-1), 0.0)

    # ➕ v3: Time masking (che 1 đoạn thời gian ngắn)
    time_mask_prob = torch.rand(batch_size, device=X_batch.device) < 0.3
    if time_mask_prob.any():
        seq_len = X_batch.shape[2]
        mask_len = seq_len // 10  # che 10% thời gian
        starts = torch.randint(0, seq_len - mask_len, (batch_size,), device=X_batch.device)
        for i in range(batch_size):
            if time_mask_prob[i]:
                X_batch[i, :, starts[i]:starts[i]+mask_len] = 0.0

    return X_batch

def preprocess_batch(X_batch, y_batch, device, training=False):
    X_batch = X_batch.to(device, non_blocking=True).float()
    y_batch = y_batch.to(device, non_blocking=True).float()

    mean = X_batch.mean(dim=2, keepdim=True)
    std = X_batch.std(dim=2, keepdim=True, unbiased=False).clamp_min_(1e-6)
    X_batch = (X_batch - mean) / std

    if training and ENABLE_BATCH_AUGMENTATION:
        X_batch = augment_batch(X_batch)

    return X_batch, y_batch

def train_epoch(model, loader, criterion, optimizer, scaler, device):
    """v3: scheduler step per epoch (not per batch) - tách khỏi hàm này."""
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc='Training', leave=False)
    for X_batch, y_batch in pbar:
        X_batch, y_batch = preprocess_batch(X_batch, y_batch, device, training=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=device.type == 'cuda'):
            outputs = model(X_batch).squeeze(1)
            loss = criterion(outputs, y_batch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        with torch.no_grad():
            probs = torch.sigmoid(outputs.float())
            preds = (probs >= 0.5).int()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

        pbar.set_postfix_str(f"loss={loss.item():.4f}")

    epoch_loss = running_loss / len(loader)
    epoch_f1 = f1_score(all_labels, all_preds, zero_division=0)
    return epoch_loss, epoch_f1

def eval_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []

    with torch.inference_mode():
        for X_batch, y_batch in loader:
            X_batch, y_batch = preprocess_batch(X_batch, y_batch, device, training=False)

            with autocast(enabled=device.type == 'cuda'):
                outputs = model(X_batch).squeeze(1)
                loss = criterion(outputs, y_batch)

            running_loss += loss.item()
            probs = torch.sigmoid(outputs.float())
            preds = (probs >= 0.5).int()

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(loader)
    epoch_f1 = f1_score(all_labels, all_preds, zero_division=0)
    return epoch_loss, epoch_f1, np.array(all_probs), np.array(all_preds), np.array(all_labels)

print('✅ Training/Eval helpers v3 đã sẵn sàng (scheduler step per epoch)')


In [ ]:
# CELL 10: Vòng lặp Huấn luyện (v3 + Early Stopping)
train_losses, val_losses = [], []
train_f1s, val_f1s = [], []
best_val_f1 = 0.0
best_epoch = 0
patience_counter = 0  # ➕ Early stopping counter

print(f'🔥 Bắt đầu huấn luyện {EPOCHS} Epochs (v3 Anti-Overfitting)...')
print(f'   Device: {DEVICE} | GPUs: {torch.cuda.device_count()}')
print(f'   Batch: {BATCH_SIZE} | LR: {LR} | WD: {WEIGHT_DECAY}')
print(f'   Scheduler: CosineAnnealingWarmRestarts (T0=10, Tmult=2)')
print(f'   Early Stopping: patience={EARLY_STOPPING_PATIENCE}')
print(f'   Loader: workers={NUM_WORKERS} | pin_memory={PIN_MEMORY} | sampler={USE_WEIGHTED_SAMPLER}')
print('=' * 70)

train_start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()

    # v3: train_epoch KHÔNG step scheduler (vì CosineAnnealing step per epoch)
    train_loss, train_f1 = train_epoch(
        model, train_loader, criterion, optimizer, scaler, DEVICE
    )
    val_loss, val_f1, val_probs, val_preds, val_labels = eval_epoch(
        model, test_loader, criterion, DEVICE
    )

    # ➕ v3: Scheduler step per epoch (không per batch)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_f1s.append(train_f1)
    val_f1s.append(val_f1)

    # In kết quả
    lr_now = optimizer.param_groups[0]['lr']
    epoch_time = time.time() - epoch_start
    gpu_msg = ''
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
        peak_gpu_mem = torch.cuda.max_memory_allocated() / (1024 ** 3)
        gpu_msg = f' | GPU: {peak_gpu_mem:.2f}GB'
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} F1: {train_f1:.4f} | '
          f'Val Loss: {val_loss:.4f} F1: {val_f1:.4f} | '
          f'LR: {lr_now:.6f}{gpu_msg} | {epoch_time:.0f}s')

    # Save best model + Early Stopping
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch + 1
        patience_counter = 0  # Reset
        torch.save(get_model_state_dict(model), os.path.join(OUTPUT_PATH, 'best_cnn_model.pth'))
        print(f'  ⭐ Best model saved! F1={best_val_f1:.4f}')
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f'\n⏹️ Early Stopping! Val F1 không cải thiện sau {EARLY_STOPPING_PATIENCE} epochs.')
            print(f'   Best: Epoch {best_epoch}, Val F1={best_val_f1:.4f}')
            break
        print(f'  ⏳ Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}')

total_train_time = time.time() - train_start_time
actual_epochs = epoch + 1
print('=' * 70)
print(f'🏆 Best: Epoch {best_epoch}/{actual_epochs}, Val F1={best_val_f1:.4f}')
print(f'⏱️ Tổng thời gian train: {total_train_time:.0f}s ({total_train_time/60:.1f} phút)')


In [ ]:
# CELL 11: Đánh giá + Biểu đồ + Tối ưu Threshold

# Load best model
best_path = os.path.join(OUTPUT_PATH, 'best_cnn_model.pth')
if os.path.exists(best_path):
    load_model_checkpoint(model, best_path, DEVICE)
    print(f'✅ Đã load model tốt nhất từ epoch {best_epoch}')

_, _, val_probs, val_preds_default, val_labels = eval_epoch(model, test_loader, criterion, DEVICE)

# ==================== THRESHOLD OPTIMIZATION ====================
print('\n🔍 Tối ưu Threshold theo F1...')
thresholds = np.arange(0.05, 0.96, 0.05)
th_results = []
for th in thresholds:
    preds_th = (val_probs >= th).astype(int)
    p = precision_score(val_labels, preds_th, zero_division=0)
    r = recall_score(val_labels, preds_th, zero_division=0)
    f = f1_score(val_labels, preds_th, zero_division=0)
    th_results.append({'threshold': th, 'precision': p, 'recall': r, 'f1': f})

th_df = pd.DataFrame(th_results).sort_values('f1', ascending=False)
print(th_df.head(10).to_string(index=False))

best_threshold = float(th_df.iloc[0]['threshold'])
print(f'\n✅ Best threshold: {best_threshold:.2f}')

# Apply best threshold
val_preds_best = (val_probs >= best_threshold).astype(int)

# ==================== 4 BIỂU ĐỒ ====================
%matplotlib inline
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('🧠 EEG Seizure Detection - 1D-CNN + BiGRU + Attention', fontsize=16, fontweight='bold')

# 1. Loss Curve
axes[0, 0].plot(train_losses, 'b-', label='Train Loss', linewidth=2)
axes[0, 0].plot(val_losses, 'r-', label='Val Loss', linewidth=2)
axes[0, 0].set_title('Training & Validation Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. F1 Curve
axes[0, 1].plot(train_f1s, 'b-', label='Train F1', linewidth=2)
axes[0, 1].plot(val_f1s, 'r-', label='Val F1', linewidth=2)
axes[0, 1].axhline(y=best_val_f1, color='g', linestyle='--', alpha=0.5, label=f'Best F1={best_val_f1:.3f}')
axes[0, 1].set_title('F1-Score qua các Epochs')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('F1-Score')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Confusion Matrix (với best threshold)
cm = confusion_matrix(val_labels, val_preds_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0],
            xticklabels=['Normal', 'Seizure'], yticklabels=['Normal', 'Seizure'])
axes[1, 0].set_title(f'Confusion Matrix (threshold={best_threshold:.2f})')
axes[1, 0].set_ylabel('Actual')
axes[1, 0].set_xlabel('Predicted')

# 4. ROC + PR Curve (dual axis)
# ROC
fpr, tpr, _ = roc_curve(val_labels, val_probs)
roc_auc = roc_auc_score(val_labels, val_probs)
axes[1, 1].plot(fpr, tpr, 'b-', label=f'ROC (AUC={roc_auc:.3f})', linewidth=2)
axes[1, 1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
# PR on same axes
prec, rec, _ = precision_recall_curve(val_labels, val_probs)
pr_auc = average_precision_score(val_labels, val_probs)
axes[1, 1].plot(rec, prec, 'r-', label=f'PR (AP={pr_auc:.3f})', linewidth=2)
axes[1, 1].set_title('ROC Curve & Precision-Recall Curve')
axes[1, 1].set_xlabel('Rate')
axes[1, 1].set_ylabel('Score')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
chart_path = os.path.join(OUTPUT_PATH, 'evaluation_dl_charts.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()

# ==================== BÁO CÁO ====================
print('\n' + '=' * 70)
print('📊 BÁO CÁO KẾT QUẢ CUỐI CÙNG')
print('=' * 70)
print(f'Best Threshold: {best_threshold:.2f}')
print(f'ROC-AUC       : {roc_auc:.4f}')
print(f'PR-AUC (AP)   : {pr_auc:.4f}')
print(f'\n{classification_report(val_labels, val_preds_best, target_names=["Normal", "Seizure"])}')

In [ ]:
# CELL 12: Đóng gói Output
import zipfile
import json

# Save metadata
metadata = {
    'architecture': 'EEG_CNN_BiGRU_Attention',
    'best_epoch': best_epoch,
    'best_val_f1': float(best_val_f1),
    'best_threshold': float(best_threshold),
    'roc_auc': float(roc_auc),
    'pr_auc': float(pr_auc),
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'pin_memory': PIN_MEMORY,
    'h5_compression': str(H5_COMPRESSION),
    'weighted_sampler': USE_WEIGHTED_SAMPLER,
    'data_parallel': use_data_parallel,
    'torch_compile': USE_COMPILE,
    'bandpass': f'{BANDPASS_LOW}-{BANDPASS_HIGH}Hz',
    'seizure_overlap': SEIZURE_OVERLAP,
    'normal_keep_prob': NORMAL_KEEP_PROB,
    'focal_alpha': FOCAL_ALPHA,
    'focal_gamma': FOCAL_GAMMA,
    'patient_split': {
        'train_patients': sorted(list(train_patients)),
        'test_patients': sorted(list(test_patients))
    }
}

meta_path = os.path.join(OUTPUT_PATH, 'model_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

# Zip
ZIP_PATH = os.path.join(OUTPUT_PATH, 'eeg_dl_outputs.zip')
files_to_zip = ['best_cnn_model.pth', 'evaluation_dl_charts.png', 'model_metadata.json']

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for fname in files_to_zip:
        fpath = os.path.join(OUTPUT_PATH, fname)
        if os.path.exists(fpath):
            zipf.write(fpath, fname)
            print(f'  ✅ {fname}')

print(f'\n📦 Đã nén: {ZIP_PATH}')

from IPython.display import FileLink, display
if os.path.exists(ZIP_PATH):
    display(FileLink(ZIP_PATH))